# Jewelry Huber Regression Pricer

Train and evaluate jewelry Huber regression baselines with and without auction estimate features.

## Configuration

In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "jewelry_huber_pricer").exists():
    PROJECT_DIR = NOTEBOOK_DIR
else:
    PROJECT_DIR = Path("Linear_Regression/jewelry/Huber_Regression").resolve()

sys.path.insert(0, str(PROJECT_DIR))

OUTPUT_DIR = PROJECT_DIR / "outputs"
WORKSPACE_ROOT = PROJECT_DIR.parents[2]
JEWELRY_DATA_DIR = WORKSPACE_ROOT / "Data" / "sampled_sets_jewelry"
TRAIN_FILE = JEWELRY_DATA_DIR / "Jewelry_2025_high_midpoint_error_remainder_training.csv"
QUARTERS = ("2025Q1", "2025Q2", "2025Q3", "2025Q4")
TEST_FILES = {
    quarter: JEWELRY_DATA_DIR / f"Jewelry_2025_high_midpoint_error_q{quarter}_sample1000.csv"
    for quarter in QUARTERS
}
TARGET_COL = "ah_price_usd"
TRAIN_BEFORE = None
TRAIN_ROWS = None
TEST_ROWS_PER_QUARTER = None
MIN_CATEGORY_COUNT = 20
VALIDATION_FRACTION = 0.10
RANDOM_STATE = 42
MAX_ITER = 1000
VERBOSE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project dir: {PROJECT_DIR}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Train file:  {TRAIN_FILE}")
for quarter, path in TEST_FILES.items():
    print(f"Test file {quarter}: {path}")

Project dir: /srv/notebooks/[6.0] ML Pricers/Linear_Regression/jewelry/Huber_Regression
Output dir:  /srv/notebooks/[6.0] ML Pricers/Linear_Regression/jewelry/Huber_Regression/outputs
Train file:  /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry/Jewelry_2025_high_midpoint_error_remainder_training.csv
Test file 2025Q1: /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry/Jewelry_2025_high_midpoint_error_q2025Q1_sample1000.csv
Test file 2025Q2: /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry/Jewelry_2025_high_midpoint_error_q2025Q2_sample1000.csv
Test file 2025Q3: /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry/Jewelry_2025_high_midpoint_error_q2025Q3_sample1000.csv
Test file 2025Q4: /srv/notebooks/[6.0] ML Pricers/Data/sampled_sets_jewelry/Jewelry_2025_high_midpoint_error_q2025Q4_sample1000.csv


## Imports

In [2]:
import pandas as pd

from jewelry_huber_pricer import (
    apply_feature_schema,
    build_feature_report,
    evaluate_by_price_bucket,
    evaluate_by_quarter,
    evaluate_price_predictions,
    fit_feature_schema,
    fit_huber_log_price_model,
    get_top_coefficients,
    huber_training_history,
    load_jewelry_data,
    make_model_frame,
    predict_price_from_log_model,
    split_train_test,
)

## Load and Split Data

In [3]:
missing_inputs = [path for path in [TRAIN_FILE, *TEST_FILES.values()] if not path.exists()]
if missing_inputs:
    raise FileNotFoundError("Missing input files:\n" + "\n".join(str(path) for path in missing_inputs))

raw = load_jewelry_data(
    train_file=TRAIN_FILE,
    test_dir=JEWELRY_DATA_DIR,
    train_before=TRAIN_BEFORE,
    quarters=QUARTERS,
    verbose=VERBOSE,
)
train_raw, test_raw = split_train_test(
    raw,
    train_rows=TRAIN_ROWS,
    test_rows_per_quarter=TEST_ROWS_PER_QUARTER,
    random_state=RANDOM_STATE,
    verbose=VERBOSE,
)

train_keys = set(train_raw["lot_url"].dropna().astype(str))
test_keys = set(test_raw["lot_url"].dropna().astype(str))
print(f"train_rows={len(train_raw):,}")
print(f"test_rows={len(test_raw):,}")
print(f"lot_url_overlap={len(train_keys & test_keys):,}")
test_raw["source_quarter"].value_counts().sort_index()

[jewelry features] Loaded Jewelry_2025_high_midpoint_error_q2025Q1_sample1000.csv: 1,000 rows, 65 columns
[jewelry features] Loaded Jewelry_2025_high_midpoint_error_q2025Q2_sample1000.csv: 1,000 rows, 65 columns
[jewelry features] Loaded Jewelry_2025_high_midpoint_error_q2025Q3_sample1000.csv: 1,000 rows, 65 columns
[jewelry features] Loaded Jewelry_2025_high_midpoint_error_q2025Q4_sample1000.csv: 1,000 rows, 65 columns
[jewelry features] Loaded jewelry frame: 9,580 rows, 65 columns
[jewelry features] Initial split: train=5,580, explicit test=4,000
[jewelry features] De-duplicated train lot_url: 5,580->5,580
[jewelry features] De-duplicated test lot_url: 4,000->4,000
[jewelry features] Removed test-overlapping lot_url values from train: 5,580->5,580
[jewelry features] Final split: train=5,580, test=4,000
train_rows=5,580
test_rows=4,000
lot_url_overlap=0


source_quarter
2025Q1    1000
2025Q2    1000
2025Q3    1000
2025Q4    1000
Name: count, dtype: int64

## Train and Evaluate Variants

In [4]:
predictions = test_raw[
    [c for c in ["ID", "lot_url", "source_file", "source_quarter", "sale_date", TARGET_COL] if c in test_raw]
].copy()
predictions = predictions.rename(columns={TARGET_COL: "actual_price_usd"})

overall_rows = []
quarter_rows = []
bucket_rows = []
history_frames = []
coef_frames = []

for include_estimates, suffix, label in [
    (False, "no_estimates", "Jewelry Huber - No Estimates"),
    (True, "with_estimates", "Jewelry Huber - With Log Estimates"),
]:
    train_frame, feature_cols, categorical_cols, numeric_cols = make_model_frame(
        train_raw,
        target_col=TARGET_COL,
        include_estimates=include_estimates,
        verbose=VERBOSE,
    )
    test_frame, _, _, _ = make_model_frame(
        test_raw,
        target_col=TARGET_COL,
        include_estimates=include_estimates,
        verbose=VERBOSE,
    )

    schema = fit_feature_schema(
        train_frame,
        feature_cols,
        categorical_cols,
        min_category_count=MIN_CATEGORY_COUNT,
        verbose=VERBOSE,
    )
    x_train = apply_feature_schema(train_frame, feature_cols, categorical_cols, schema, verbose=VERBOSE)
    x_test = apply_feature_schema(test_frame, feature_cols, categorical_cols, schema, verbose=VERBOSE)

    model = fit_huber_log_price_model(
        x_train,
        train_frame["log_target_price_usd"],
        categorical_cols=categorical_cols,
        numeric_cols=numeric_cols,
        validation_fraction=VALIDATION_FRACTION,
        random_state=RANDOM_STATE,
        max_iter=MAX_ITER,
        verbose=VERBOSE,
    )
    pred = predict_price_from_log_model(model, x_test)
    predictions[f"huber_pred_{suffix}_usd"] = pred

    overall_rows.append(evaluate_price_predictions(test_frame["target_price_usd"], pred, label=label))
    quarter_rows.append(evaluate_by_quarter(test_frame, pred, actual_col="target_price_usd", label=label))
    bucket_rows.append(evaluate_by_price_bucket(test_frame, pred, actual_col="target_price_usd", label=label))

    feature_report = build_feature_report(train_frame, feature_cols, categorical_cols, schema=schema)
    feature_report.to_csv(OUTPUT_DIR / f"jewelry_huber_feature_report_{suffix}.csv", index=False)

    history = huber_training_history(model)
    if not history.empty:
        history.insert(0, "model", label)
        history_frames.append(history)

    coefs = get_top_coefficients(model, top_n=40)
    coefs.insert(0, "model", label)
    coef_frames.append(coefs)

overall = pd.DataFrame(overall_rows)
quarter = pd.concat(quarter_rows, ignore_index=True)
bucket = pd.concat(bucket_rows, ignore_index=True)
overall

[jewelry features] Building jewelry model frame from 5,580 rows using target_col='ah_price_usd', include_estimates=False
[jewelry features] Filtered invalid/non-positive targets: 5,580->5,580
[jewelry features] Engineered 55 features (29 categorical, 26 numeric).
[jewelry features] Building jewelry model frame from 4,000 rows using target_col='ah_price_usd', include_estimates=False
[jewelry features] Filtered invalid/non-positive targets: 4,000->4,000
[jewelry features] Engineered 55 features (29 categorical, 26 numeric).
[jewelry features] Fitting jewelry feature schema with min_category_count=20
[jewelry features]   categorical auction_house: 7 kept levels from 5 observed
[jewelry features]   categorical location: 11 kept levels from 10 observed
[jewelry features]   categorical product_name_clean: 9 kept levels from 3,796 observed
[jewelry features]   categorical brand_clean: 21 kept levels from 501 observed
[jewelry features]   categorical collection_clean: 2 kept levels from 104 ob

/home/vann/.local/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


[jewelry features] Applying jewelry feature schema to 4,000 rows and 55 features
[huber model] Training Huber on 5,580 rows and 55 features
[huber model] Alpha grid: [0.0001, 0.001, 0.01]
[huber model] Epsilon grid: [1.35, 1.75]
[huber model] Running Huber diagnostics: train_fit=5,022, validation=558, candidates=6, eval_metric=log_RMSE
[huber model] [01/06] alpha=0.0001 epsilon=1.35 log_RMSE=1.35035 log_MAE=0.76364 RMSE=2376065360755.32 MAE=100586910535.04 outlier_fraction=0.368
[huber model] [02/06] alpha=0.0001 epsilon=1.75 log_RMSE=1.24339 log_MAE=0.76095 RMSE=51910333631.29 MAE=2197582523.37 outlier_fraction=0.128
[huber model] [03/06] alpha=0.001 epsilon=1.35 log_RMSE=1.34728 log_MAE=0.76339 RMSE=2143614971869.95 MAE=90746500900.61 outlier_fraction=0.368
[huber model] [04/06] alpha=0.001 epsilon=1.75 log_RMSE=1.25502 log_MAE=0.76187 RMSE=80580655840.06 MAE=3411294042.90 outlier_fraction=0.128
[huber model] [05/06] alpha=0.01 epsilon=1.35 log_RMSE=1.34875 log_MAE=0.76334 RMSE=22668

/home/vann/.local/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


[jewelry features] Applying jewelry feature schema to 4,000 rows and 59 features
[huber model] Training Huber on 5,580 rows and 59 features
[huber model] Alpha grid: [0.0001, 0.001, 0.01]
[huber model] Epsilon grid: [1.35, 1.75]
[huber model] Running Huber diagnostics: train_fit=5,022, validation=558, candidates=6, eval_metric=log_RMSE


...

/home/vann/.local/lib/python3.11/site-packages/sklearn/linear_model/_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


[huber model] [06/06] alpha=0.01 epsilon=1.75 log_RMSE=0.30532 log_MAE=0.21527 RMSE=59343.11 MAE=10077.23 outlier_fraction=0.121
[huber model] Selected alpha=0.001, epsilon=1.75


/home/vann/.local/lib/python3.11/site-packages/sklearn/linear_model/_huber.py:348: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


[huber model] Final fit: alpha=0.001, epsilon=1.75, n_iter=1000, outlier_fraction=0.121


,model,n,MAE,RMSE,MAPE,R2
0,Jewelry Huber - No Estimates,4000,77849.915541,1.465616e+06,3.781127,-19.234347
1,Jewelry Huber - With Log Estimates,4000,30616.117192,1.952411e+05,0.560175,0.640920


## Write Outputs

In [5]:
predictions.to_csv(OUTPUT_DIR / "jewelry_huber_test_predictions.csv", index=False)
overall.to_csv(OUTPUT_DIR / "jewelry_huber_overall_metrics.csv", index=False)
quarter.to_csv(OUTPUT_DIR / "jewelry_huber_quarter_metrics.csv", index=False)
bucket.to_csv(OUTPUT_DIR / "jewelry_huber_price_bucket_metrics.csv", index=False)
if history_frames:
    pd.concat(history_frames, ignore_index=True).to_csv(OUTPUT_DIR / "jewelry_huber_training_history.csv", index=False)
pd.concat(coef_frames, ignore_index=True).to_csv(OUTPUT_DIR / "jewelry_huber_top_coefficients.csv", index=False)

print(f"Wrote outputs to {OUTPUT_DIR}")

Wrote outputs to /srv/notebooks/[6.0] ML Pricers/Linear_Regression/jewelry/Huber_Regression/outputs
